# 19 — National Hydro Network (NHN) Integration

## Purpose

This notebook explores integration of the **National Hydro Network (NHN)** into the GeoCANOE geospatial workflow.

The immediate objective is to use national hydrographic data to characterize where GeoCANOE candidate graph edges intersect lakes and other relevant surface-water features. These intersections can later be converted into **basemap-specific physical edge attributes** and subsequently used by technology-specific transport cost models.

The NHN is therefore treated as a **geospatial evidence layer**, not as a direct CANOE/TEMOA model input.

---

## Source

**Dataset:** National Hydro Network (NHN)  
**Provider:** Natural Resources Canada (NRCan)

Government of Canada dataset page:

<https://open.canada.ca/data/en/dataset/a4b190fe-e090-4e6d-881e-b87956c07977>

Canada-wide GeoPackage distribution:

<https://ftp.maps.canada.ca/pub/nrcan_rncan/vector/geobase_nhn_rhn/gpkg_en/CA/>

NRCan provides several national GeoPackage archives in this directory, including hydrographic features, hydrographic events, network topology, named features, and NHN partition information. The national hydrographic-feature archive is distributed as:

`rhn_nhn_hhyd.gpkg.zip`

The archive is approximately **14 GB compressed**, so the acquisition and preprocessing workflow will need to account explicitly for dataset size. :contentReference[oaicite:0]{index=0}

---

## Initial scope

The first implementation will focus on identifying and extracting the **polygonal waterbody information** required to evaluate candidate infrastructure corridors.

The exploratory workflow will:

1. acquire the relevant NHN national GeoPackage;
2. inspect its internal layers, schema, geometry types, CRS, and attributes;
3. identify the hydrographic features appropriate for representing lakes and other significant surface-water bodies;
4. determine the minimum subset of NHN information required by GeoCANOE;
5. standardize the selected hydrographic data into a reusable processed layer;
6. overlay that layer with GeoCANOE graph edges generated from a selected basemap;
7. calculate physical edge attributes describing interaction with water.

---

## Intended model relationship

The NHN source itself is independent of GeoCANOE basemap resolution.

However, water-intersection metrics are **derived properties of a particular graph edge** and therefore depend on the basemap and graph configuration used to construct that edge.

The intended workflow is:

`NHN source → standardized hydrographic layer → selected GeoCANOE graph → edge-waterbody overlay → edge attributes → transport cost model`

Potential edge attributes include:

- total edge length;
- total length intersecting water;
- fraction of edge length intersecting water;
- number of distinct waterbody intersections;
- maximum continuous water crossing;
- identifiers or classifications of intersected water bodies.

These are physical spatial attributes rather than economic assumptions.

---

## Design principle

The hydrographic preprocessing layer should answer:

> **How does this candidate graph edge interact with surface water?**

It should not directly answer:

> **How much should this interaction cost?**

The resulting physical attributes can instead be passed downstream to technology-specific cost formulations.

For example, the same water crossing could impose different engineering and economic consequences for:

- CO₂ pipelines;
- hydrogen pipelines;
- transmission corridors;
- roads;
- rail infrastructure.

Separating **geospatial evidence**, **derived edge characteristics**, and **technology-specific cost assumptions** allows the NHN layer to remain reusable across multiple GeoCANOE transport representations.

---

## Notebook status

This notebook is exploratory.

The immediate task is to understand the structure of the national NHN GeoPackage and identify the smallest useful hydrographic subset before designing the permanent Bronze acquisition and Silver preprocessing workflow.

In [ ]:
# ---------------------------------------------------------------------------
# Cell 2 — NHN national GeoPackage source configuration
# ---------------------------------------------------------------------------

from pathlib import Path
import requests

from bs4 import BeautifulSoup
from tqdm.auto import tqdm

import sqlite3
import pandas as pd
import geopandas as gpd
from shapely.geometry import box


from geocanoe.paths import find_project_root


# ---------------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------------

PROJECT_ROOT = find_project_root()

RAW_NHN = (
    PROJECT_ROOT
    / "data_files"
    / "raw"
    / "nhn"
)

RAW_NHN.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# Source definitions
# ---------------------------------------------------------------------------

NHN_SOURCE_PAGE = (
    "https://open.canada.ca/data/en/dataset/"
    "a4b190fe-e090-4e6d-881e-b87956c07977"
)

NHN_GPKG_BASE_URL = (
    "https://ftp.maps.canada.ca/pub/nrcan_rncan/"
    "vector/geobase_nhn_rhn/gpkg_en/CA"
)

NHN_HHYD_ARCHIVE = "rhn_nhn_hhyd.gpkg.zip"

NHN_HHYD_URL = (
    f"{NHN_GPKG_BASE_URL}/{NHN_HHYD_ARCHIVE}"
)

NHN_HHYD_LOCAL_PATH = (
    RAW_NHN
    / NHN_HHYD_ARCHIVE
)


# ---------------------------------------------------------------------------
# Source availability check
# ---------------------------------------------------------------------------

response = requests.head(
    NHN_HHYD_URL,
    timeout=120,
    allow_redirects=True,
)

response.raise_for_status()

content_length = response.headers.get("Content-Length")

print("NHN national hydrographic GeoPackage source reachable.")
print(f"Status code: {response.status_code}")
print(f"Archive: {NHN_HHYD_ARCHIVE}")
print(f"Bronze destination: {NHN_HHYD_LOCAL_PATH}")

if content_length is not None:
    size_gb = int(content_length) / 1e9
    print(f"Reported archive size: {size_gb:.2f} GB")
else:
    print("Archive size not reported by server.")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 3 — Inspect NHN national GeoPackage distribution metadata
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Inspect national GeoPackage directory
# ---------------------------------------------------------------------------

directory_response = requests.get(
    f"{NHN_GPKG_BASE_URL}/",
    timeout=120,
)

directory_response.raise_for_status()

directory_soup = BeautifulSoup(
    directory_response.text,
    "html.parser",
)

directory_files = []

for link in directory_soup.find_all("a"):
    href = link.get("href")

    if not href:
        continue

    href = href.strip()

    if href in {"../", "./"} or href.startswith("?"):
        continue

    directory_files.append(href)


print("Files available in national NHN GeoPackage distribution:\n")

for filename in directory_files:
    print(f"  {filename}")

In [ ]:
# ---------------------------------------------------------------------------
# Read national NHN distribution documentation
# ---------------------------------------------------------------------------

NHN_README_URL = (
    f"{NHN_GPKG_BASE_URL}/lisezmoi_readme.txt"
)

readme_response = requests.get(
    NHN_README_URL,
    timeout=120,
)

readme_response.raise_for_status()

nhn_readme = readme_response.text

print(nhn_readme)

In [ ]:
# ---------------------------------------------------------------------------
# Cell 4 — Inspect NHN national GeoPackage README
# ---------------------------------------------------------------------------

NHN_README_URL = (
    f"{NHN_GPKG_BASE_URL}/lisezmoi_readme.txt"
)

readme_response = requests.get(
    NHN_README_URL,
    timeout=120,
)

readme_response.raise_for_status()

nhn_readme = readme_response.text

print(nhn_readme)

# ---------------------------------------------------------------------------
# Extract the hydrographic-feature GeoPackage section
# ---------------------------------------------------------------------------

lines = nhn_readme.splitlines()

hhyd_lines = []
capture = False

for line in lines:
    if "rhn_nhn_hhyd.gpkg.zip" in line:
        capture = True

    elif capture and line.strip().endswith(".gpkg.zip"):
        break

    if capture:
        hhyd_lines.append(line)

print("\n".join(hhyd_lines))

In [ ]:
# ---------------------------------------------------------------------------
# Cell 5 — Download national NHN hydrographic GeoPackage archive
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Download configuration
# ---------------------------------------------------------------------------

CHUNK_SIZE = 1024 * 1024  # 1 MB


# ---------------------------------------------------------------------------
# Download archive
# ---------------------------------------------------------------------------

if NHN_HHYD_LOCAL_PATH.exists():
    print(f"Archive already exists:")
    print(f"  {NHN_HHYD_LOCAL_PATH}")

else:
    print("Downloading national NHN hydrographic archive...")
    print(f"Source:      {NHN_HHYD_URL}")
    print(f"Destination: {NHN_HHYD_LOCAL_PATH}")
    print()

    with requests.get(
        NHN_HHYD_URL,
        stream=True,
        timeout=120,
    ) as response:
        response.raise_for_status()

        total_bytes = int(
            response.headers.get("Content-Length", 0)
        )

        with (
            NHN_HHYD_LOCAL_PATH.open("wb") as file,
            tqdm(
                total=total_bytes,
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                desc=NHN_HHYD_ARCHIVE,
            ) as progress,
        ):
            for chunk in response.iter_content(
                chunk_size=CHUNK_SIZE
            ):
                if chunk:
                    file.write(chunk)
                    progress.update(len(chunk))


# ---------------------------------------------------------------------------
# Confirm local archive
# ---------------------------------------------------------------------------

archive_size_gb = (
    NHN_HHYD_LOCAL_PATH.stat().st_size
    / 1e9
)

print()
print("Download complete.")
print(f"Archive size: {archive_size_gb:.2f} GB")
print(f"Path: {NHN_HHYD_LOCAL_PATH}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 6 — Extract national NHN hydrographic GeoPackage
# ---------------------------------------------------------------------------

import zipfile


# ---------------------------------------------------------------------------
# Inspect archive contents
# ---------------------------------------------------------------------------

with zipfile.ZipFile(NHN_HHYD_LOCAL_PATH) as archive:
    archive_members = archive.namelist()

print(f"Files in archive: {len(archive_members):,}")

for member in archive_members:
    print(f"  {member}")


# ---------------------------------------------------------------------------
# Identify GeoPackage member
# ---------------------------------------------------------------------------

gpkg_members = [
    member
    for member in archive_members
    if member.lower().endswith(".gpkg")
]

if len(gpkg_members) != 1:
    raise ValueError(
        "Expected exactly one GeoPackage in the NHN archive, "
        f"found {len(gpkg_members)}: {gpkg_members}"
    )

gpkg_member = gpkg_members[0]

NHN_HHYD_GPKG_PATH = (
    RAW_NHN
    / Path(gpkg_member).name
)


# ---------------------------------------------------------------------------
# Extract GeoPackage
# ---------------------------------------------------------------------------

if NHN_HHYD_GPKG_PATH.exists():
    print()
    print("GeoPackage already extracted:")
    print(f"  {NHN_HHYD_GPKG_PATH}")

else:
    print()
    print("Extracting national NHN hydrographic GeoPackage...")

    with zipfile.ZipFile(NHN_HHYD_LOCAL_PATH) as archive:
        with archive.open(gpkg_member) as source:
            with NHN_HHYD_GPKG_PATH.open("wb") as destination:
                while True:
                    chunk = source.read(1024 * 1024)

                    if not chunk:
                        break

                    destination.write(chunk)

    print("Extraction complete.")


# ---------------------------------------------------------------------------
# Report extracted size
# ---------------------------------------------------------------------------

gpkg_size_gb = (
    NHN_HHYD_GPKG_PATH.stat().st_size
    / 1e9
)

print()
print(f"GeoPackage size: {gpkg_size_gb:.2f} GB")
print(f"Path: {NHN_HHYD_GPKG_PATH}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 9 — Inspect NHN waterbody classification fields
# ---------------------------------------------------------------------------

NHN_WATERBODY_LAYER = "nhn_hhyd_Waterbody_2"

classification_fields = [
    "water_definition",
    "permanency",
    "isolated",
    "code_spec",
    "completely_cover",
    "acquisition_technique",
    "provider",
]

with sqlite3.connect(NHN_HHYD_GPKG_PATH) as connection:

    for field in classification_fields:

        print()
        print("=" * 78)
        print(field)
        print("=" * 78)

        values = pd.read_sql_query(
            f"""
            SELECT
                "{field}" AS value,
                COUNT(*) AS feature_count
            FROM "{NHN_WATERBODY_LAYER}"
            GROUP BY "{field}"
            ORDER BY feature_count DESC
            """,
            connection,
        )

        display(values)

In [ ]:
# ---------------------------------------------------------------------------
# Cell 9 — Inspect NHN waterbody classification fields
# ---------------------------------------------------------------------------

classification_fields = [
    "water_definition",
    "permanency",
    "isolated",
    "code_spec",
    "completely_cover",
    "acquisition_technique",
    "provider",
]


with sqlite3.connect(NHN_HHYD_GPKG_PATH) as connection:

    for field in classification_fields:

        print()
        print("=" * 78)
        print(field)
        print("=" * 78)

        values = pd.read_sql_query(
            f"""
            SELECT
                "{field}" AS value,
                COUNT(*) AS feature_count
            FROM "{NHN_WATERBODY_LAYER}"
            GROUP BY "{field}"
            ORDER BY feature_count DESC
            """,
            connection,
        )

        display(values)

In [ ]:
# ---------------------------------------------------------------------------
# Cell 10 — Cross-tabulate NHN water definition and permanency
# ---------------------------------------------------------------------------

with sqlite3.connect(NHN_HHYD_GPKG_PATH) as connection:

    waterbody_class_summary = pd.read_sql_query(
        f"""
        SELECT
            water_definition,
            permanency,
            COUNT(*) AS feature_count
        FROM "{NHN_WATERBODY_LAYER}"
        GROUP BY
            water_definition,
            permanency
        ORDER BY
            water_definition,
            permanency
        """,
        connection,
    )

display(waterbody_class_summary)

In [ ]:
# ---------------------------------------------------------------------------
# Cell 11 — Crosswalk water_definition and code_spec
# ---------------------------------------------------------------------------

with sqlite3.connect(NHN_HHYD_GPKG_PATH) as connection:

    water_definition_codes = pd.read_sql_query(
        f"""
        SELECT
            water_definition,
            code_spec,
            COUNT(*) AS feature_count
        FROM "{NHN_WATERBODY_LAYER}"
        GROUP BY
            water_definition,
            code_spec
        ORDER BY
            water_definition,
            feature_count DESC
        """,
        connection,
    )

display(water_definition_codes)

In [ ]:
# ---------------------------------------------------------------------------
# Inspect representative records for each water_definition class
# ---------------------------------------------------------------------------

with sqlite3.connect(NHN_HHYD_GPKG_PATH) as connection:

    examples = pd.read_sql_query(
        f"""
        WITH ranked AS (
            SELECT
                id,
                water_definition,
                permanency,
                isolated,
                code_spec,
                dataset_name,
                provider,
                lakeid_1,
                lakeid_2,
                lakename_1,
                lakename_2,
                rivid_1,
                rivid_2,
                rivname_1,
                rivname_2,
                ROW_NUMBER() OVER (
                    PARTITION BY water_definition
                    ORDER BY id
                ) AS rn
            FROM "{NHN_WATERBODY_LAYER}"
        )
        SELECT *
        FROM ranked
        WHERE rn <= 5
        ORDER BY
            water_definition,
            rn
        """,
        connection,
    )

display(examples)